In [7]:
# autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Libraries

In [8]:
import plotly.express as px
from data.load_data import load_wisconsin_dataset
from data.split_data import DataSplitter
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import umap



# Data description

## Wisconsin Breast Cancer Dataset

The Wisconsin Breast Cancer Dataset is a widely used dataset for binary classification problems in machine learning. It contains features computed from digitized images of fine needle aspirate (FNA) of breast masses. These features describe characteristics of the cell nuclei present in the images.

The objective of the dataset is to classify tumors into two categories:

- **Malignant (M) - Class 0**: cancerous tumors
- **Benign (B) - Class 1**: non-cancerous tumors

### Dataset Size

- **Total samples:** 569
- **Number of features:** 30 numerical features
- **Target variable:** Diagnosis (Malignant or Benign)

The dataset is moderately imbalanced, with approximately:
- **357 benign cases**
- **212 malignant cases**

### Feature Description

The 30 features represent measurements of cell nuclei characteristics. These measurements are computed from the images and grouped into three categories:

1. **Mean values** – average measurement across the nuclei
2. **Standard error (SE)** – variability of the measurement
3. **Worst values** – largest value observed

Examples of features include:

- **Radius** – mean distance from the center to points on the perimeter
- **Texture** – standard deviation of gray-scale values
- **Perimeter** – length of the boundary of the nucleus
- **Area** – area of the nucleus
- **Smoothness** – local variation in radius lengths
- **Compactness** – perimeter² / area − 1
- **Concavity** – severity of concave portions of the contour
- **Symmetry** – symmetry of the nucleus
- **Fractal dimension** – measure of contour complexity

**Reference:**  
https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-wisconsin-diagnostic-dataset


# Data loading

In [9]:
wisconsin_tuple = load_wisconsin_dataset()
df = wisconsin_tuple[0]
target = wisconsin_tuple[1]

In [10]:
print(" Wisconsin Breast Cancer Dataset Preview")

print("First rows of the feature dataset:")
display(df.head())


print("First rows of the target variable (diagnosis):")
display(target.head())


 Wisconsin Breast Cancer Dataset Preview
First rows of the feature dataset:


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


First rows of the target variable (diagnosis):


0    0
1    0
2    0
3    0
4    0
Name: target, dtype: int64

In [11]:
data_splitter = DataSplitter()
X_train, X_test, y_train, y_test = data_splitter.split(
    X=df,
    y=target,
    test_size=0.15,
    val_size=None,
    stratify=True,
)

# Data analysis

## Distribution of diagnosis classes

In [12]:
print("Distribution of diagnosis classes")

target_counts = target.value_counts().reset_index()
target_counts.columns = ["Diagnosis", "Count"]

fig = px.bar(
    target_counts,
    x="Diagnosis",
    y="Count",
    color="Diagnosis",
    title="Distribution of Tumor Diagnosis",
    text="Count"
)

fig.show()


Distribution of diagnosis classes


## Mutual information analysis

In [ ]:
print("Computing Mutual Information between features and target")

mi = mutual_info_classif(X_train, y_train)

mi_df = pd.DataFrame({
    "feature": X_train.columns,
    "mutual_information": mi
}).sort_values("mutual_information", ascending=True)  

fig = px.bar(
    mi_df,
    x="mutual_information",
    y="feature",
    orientation="h",
    title="Feature Importance using Mutual Information",
    height=800 
)

fig.update_layout(
    yaxis=dict(
        title="Feature",
        automargin=True
    ),
    xaxis_title="Mutual Information",
    yaxis_title="Feature",
)

fig.show()


Computing Mutual Information between features and target


The Mutual Information analysis shows that several geometric features of the tumor provide the highest predictive information for the diagnosis in the Breast Cancer Wisconsin Diagnostic Dataset. In particular, variables such as worst perimeter, worst area, worst radius, and concave points exhibit the largest mutual information values, indicating that they contribute the most to reducing uncertainty about whether a tumor is benign or malignant.


## Correlation analysis

In [14]:
print("Kendall correlation matrix between features")

corr = X_train.corr(method='kendall')

fig = px.imshow(
    corr,
    color_continuous_scale="viridis",
    title="Feature Correlation Matrix",
    text_auto=".1f" 
)

fig.update_traces(
    textfont_size=10 
)

fig.update_layout(
    height=1000,
    width=1000
)

fig.show()


Kendall correlation matrix between features


Strong positive correlations are observed among geometric measurements such as radius, perimeter, and area, as well as between their corresponding worst values, indicating that these variables consistently increase together and capture closely related aspects of tumor size. Similarly, features describing boundary irregularity—such as concavity, concave points, and compactness—show moderate to high associations. In contrast, variables such as texture, symmetry, and fractal dimension exhibit weaker correlations with most other features, implying that they provide more independent information.


## PCA

In [15]:
print("PCA 3D visualization")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(
    X_pca,
    columns=["PC1", "PC2", "PC3"]
)

pca_df["diagnosis"] = y_train.values

fig = px.scatter_3d(
    pca_df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="diagnosis",
    title="3D PCA Projection of Breast Cancer Dataset",
    opacity=0.8
)

fig.show()


PCA 3D visualization


The 3D PCA projection shows a clear global separation between the two diagnostic classes, with both groups forming relatively compact and distinguishable clusters along the first principal components. Although PCA is a linear dimensionality reduction method, the visualization suggests that most of the discriminative information is captured in the first three components, resulting in minimal overlap between classes and only a small transition region at the boundary. This indicates that the dataset exhibits near-linear separability in the projected space, suggesting that relatively simple linear or low-complexity models could perform well on this task.

## UMAP

In [16]:
print("UMAP dimensionality reduction")

reducer = umap.UMAP(
    n_components=3,
    random_state=42,
    metric='manhattan',
    output_metric='manhattan',
    n_jobs=1
)

X_umap = reducer.fit_transform(X_scaled)

umap_df = pd.DataFrame(
    X_umap,
    columns=["UMAP1", "UMAP2", "UMAP3"]
)

umap_df["diagnosis"] = y_train.values

fig = px.scatter_3d(
    umap_df,
    x="UMAP1",
    y="UMAP2",
    z="UMAP3",
    color="diagnosis",
    title="3D UMAP Projection of Breast Cancer Dataset",
    opacity=0.8
)

fig.show()


UMAP dimensionality reduction


The 3D UMAP projection shows a clear and well-defined separation between the two diagnosis classes. The samples form two compact and highly cohesive clusters, with minimal overlap between them in the embedded space. The embedding suggests that the dataset exhibits a high degree of class separability in a low-dimensional representation, with well-clustered groups and limited ambiguity at the boundary between classes.